In [ ]:
import os
import random
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import Model
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.manifold import TSNE


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = "/kaggle/input/afib-data/lead"
SPLIT_DIR = "/kaggle/working/split_data"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-4

CLASSES = ["AF", "NAF", "PAF"]


if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

for split in ["train", "val", "test"]:
    for cls in CLASSES:
        os.makedirs(os.path.join(SPLIT_DIR, split, cls), exist_ok=True)


all_images = []

for cls in CLASSES:
    cls_path = os.path.join(DATA_DIR, cls, "cqt")

    for file in os.listdir(cls_path):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            all_images.append(
                (os.path.join(cls_path, file), cls)
            )


random.shuffle(all_images)

class_images = {cls: [] for cls in CLASSES}

for path, cls in all_images:
    class_images[cls].append(path)


for cls in CLASSES:
    images = class_images[cls]
    n = len(images)

    train_end = int(0.70 * n)
    val_end = int(0.90 * n)

    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    for src in train_images:
        shutil.copy2(
            src,
            os.path.join(SPLIT_DIR, "train", cls, os.path.basename(src))
        )

    for src in val_images:
        shutil.copy2(
            src,
            os.path.join(SPLIT_DIR, "val", cls, os.path.basename(src))
        )

    for src in test_images:
        shutil.copy2(
            src,
            os.path.join(SPLIT_DIR, "test", cls, os.path.basename(src))
        )


train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0
)


train_generator = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


class SimAM(layers.Layer):
    def __init__(self, e_lambda=1e-4, **kwargs):
        super().__init__(**kwargs)
        self.e_lambda = e_lambda

    def call(self, inputs):
        mean = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
        squared = tf.square(inputs - mean)

        n = tf.cast(
            tf.shape(inputs)[1] * tf.shape(inputs)[2] - 1,
            tf.float32
        )

        variance = tf.reduce_sum(squared, axis=[1, 2], keepdims=True) / (
            n + self.e_lambda
        )

        energy = squared / (
            4.0 * (variance + self.e_lambda)
        ) + 0.5

        attention = tf.sigmoid(1.0 / energy)

        return inputs * attention


class CSA(layers.Layer):
    def __init__(self, filters, **kwargs):
        super().__init__(**kwargs)

        self.filters = filters

        self.conv1 = layers.DepthwiseConv2D(
            3,
            padding="same"
        )

        self.conv2 = layers.DepthwiseConv2D(
            (1, 5),
            padding="same"
        )

        self.conv3 = layers.DepthwiseConv2D(
            (5, 1),
            padding="same"
        )

        self.pointwise = layers.Conv2D(
            filters,
            1,
            padding="same"
        )

        self.bn = layers.BatchNormalization()
        self.relu = layers.ReLU()

    def call(self, inputs):
        x1 = self.conv1(inputs)
        x2 = self.conv2(inputs)
        x3 = self.conv3(inputs)

        x = tf.concat([x1, x2, x3], axis=-1)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.relu(x)

        return x


def build_model():
    inputs = layers.Input(shape=(224, 224, 3))

    base_model = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )

    base_model.trainable = False

    x = base_model.output

    x = layers.AveragePooling2D(
        pool_size=(2, 2)
    )(x)

    x = layers.SeparableConv2D(
        512,
        kernel_size=(5, 5),
        padding="same",
        activation="relu"
    )(x)

    x = CSA(512)(x)

    x = SimAM()(x)

    x = layers.GlobalAveragePooling2D()(x)

    embedding = layers.Dense(
        128,
        activation="relu",
        name="embedding"
    )(x)

    x = layers.Dropout(0.3)(embedding)

    outputs = layers.Dense(
        3,
        activation="softmax",
        name="classifier"
    )(x)

    model = Model(inputs, outputs)

    embedding_model = Model(
        inputs,
        embedding
    )

    return model, embedding_model


model, embedding_model = build_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LR
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-7
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        "/kaggle/working/best_model.keras",
        monitor="val_accuracy",
        save_best_only=True
    )
]


history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)


plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()


test_generator.reset()

predictions = model.predict(test_generator)

y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print(
    classification_report(
        y_true,
        y_pred,
        target_names=CLASSES,
        digits=4
    )
)


cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(range(len(CLASSES)), CLASSES)
plt.yticks(range(len(CLASSES)), CLASSES)

for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


test_generator.reset()

embeddings = embedding_model.predict(
    test_generator,
    verbose=1
)

tsne = TSNE(
    n_components=2,
    random_state=SEED,
    perplexity=30
)

embedding_2d = tsne.fit_transform(embeddings)


plt.figure(figsize=(8, 6))

for i, cls in enumerate(CLASSES):
    idx = y_true == i

    plt.scatter(
        embedding_2d[idx, 0],
        embedding_2d[idx, 1],
        label=cls,
        alpha=0.6
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE Visualization of Learned Embeddings")
plt.legend()
plt.tight_layout()
plt.show()


results = model.evaluate(
    test_generator,
    verbose=1
)

print("Test Loss:", results[0])
print("Test Accuracy:", results[1])


model.save(
    "/kaggle/working/af_paf_mobilenetv2_model.keras"
)